In [0]:
# Notebook ; gold_dim_time
# Gold Layer: dim_time Dimension Table
# Domain: Cross-Domain Reference
# Source: silver.time
# Target: gold.dim_time
# Description: Derives SK_TimeID surrogate key from TimeValue (HHmmss format),
#              computes time dimension attributes (hour, minute, second descriptors),
#              adds MarketHoursFlag and OfficeHoursFlag, filters to valid 86,400 seconds,
#              and writes the time-of-day dimension to the Gold star schema layer.
#              Run_id carry-forwarded from Silver.

In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

#Initialize logger
logger = logging.getLogger("SilverToGold_DimTime")
logger.setLevel(logging.INFO)

In [0]:
# Paths & Tables
source_silver_table = "charles_schwab_retailbrokerage_dev_team_lemma.silver.time"
target_gold_table = "charles_schwab_retailbrokerage_dev_team_lemma.gold.dim_time"

In [0]:
def build_dim_time():
    logger.info("Silver to Gold transformation for dim_time")

    try:
        #Read silver data
        silver_df = spark.table(source_silver_table)
        #Surrogate key
        gold_df = (
            silver_df
            .withColumn(
                "SK_TimeID",
                (col("HourID") * 10000 + col("MinuteID") * 100 + col("SecondID")).cast(IntegerType())
            )
            .withColumn("_load_ts", current_timestamp())
        )

        #Schema Columns
        final_gold_df = gold_df.select(
            "SK_TimeID",
            "TimeValue",
            "HourID",
            "HourDesc",
            "MinuteID",
            "MinuteDesc",
            "SecondID",
            "SecondDesc",
            "MarketHoursFlag",
            "OfficeHoursFlag",
            "_batch",
            "_load_ts"
        )
        #Gold Delta Table
        (
            final_gold_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_gold_table)
        )
        logger.info("Successfully transformed and wrote dim_time")
        return True
    except Exception as e:
        logger.error(f"Pipeline failed for dim_time {str(e)}")
        raise e 

table_built = build_dim_time()

In [0]:
if table_built:
    try:
        gold_df = spark.table(target_gold_table)
        actual_count = gold_df.count()
        expected_count = 86400

        logger.info("Gold Reconciliation Summary")
        logger.info("Target Table : dim_time")
        logger.info(f"Expected Rows : {expected_count}")
        logger.info(f"Actual Rows : {actual_count}")

        if actual_count == expected_count :
            logger.info("Success")
        else:
            logger.info("Failure")

        display(f"Actual Count : {actual_count}")
        display(f"Expected Count : {expected_count}")
    except Exception as e:
        logger.error(f"Reconciliation failed for dim_time {str(e)}")
        raise e